# Group-Aware & Extreme Row Slicing (`head` & `tail`)

[← Back to Master Reference](../../pytae.md)

`head()` and `tail()` select the first or last $N$ rows (or proportion of rows via `prop=`), with boundary tie-breaking (`with_ties=True`) and complete group awareness via `pt.by()`.

In accordance with Pytae's *"One and Only One Obvious Way"*, extreme row selection is achieved by composing `arrange()` with `head()` or `tail()`, eliminating the need for niche specialized verbs like `pick`:

| Method | Key Parameters | Purpose |
|---|---|---|
| `df.head(n=5, prop=None, with_ties=False)` | `n`, `prop`, `with_ties` | First $N$ rows (or fraction) |
| `df.tail(n=5, prop=None, with_ties=False)` | `n`, `prop`, `with_ties` | Last $N$ rows (or fraction) |
| `df.arrange(...).head(n)` | — | Extreme top rows overall |
| `df.arrange(...).tail(n)` | — | Extreme bottom rows overall |
| `df.by(...).arrange(...).head(n)` | `*cols` | Extreme rows partitioned per group |


In [1]:
import sys, os
_src = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'src'))
if not os.path.exists(_src): _src = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if _src not in sys.path: sys.path.insert(0, _src)

import pytae as pt

penguins = pt.sample_data['penguins']
tips = pt.sample_data['tips']


## 1. Composing `arrange()` with `head()` — Top N Records

Retrieve rows with the highest values by sorting descending and slicing the head.


In [2]:
# Top 3 heaviest penguins overall
(
    penguins
    .arrange("body_mass_g desc")
    .head(3)
    .select("species", "island", "body_mass_g")
)


species,island,body_mass_g
str,str,f64
"""Gentoo""","""Biscoe""",6300.0
"""Gentoo""","""Biscoe""",6050.0
"""Gentoo""","""Biscoe""",6000.0


## 2. Composing `arrange()` with `head()` or `tail()` — Bottom N Records

Retrieve rows with the lowest values by sorting ascending or taking the tail.


In [3]:
# 3 lightest penguins overall
(
    penguins
    .arrange("body_mass_g")
    .head(3)
    .select("species", "island", "body_mass_g")
)


species,island,body_mass_g
str,str,f64
"""Chinstrap""","""Dream""",2700.0
"""Adelie""","""Biscoe""",2850.0
"""Adelie""","""Biscoe""",2850.0


## 3. Group-Aware Slicing (`df.by()`)

When preceded by `.by(...)`, `head()` and `tail()` extract rows **per group**, returning a clean, ungrouped DataFrame.


In [4]:
# Heaviest penguin for each species
(
    penguins
    .by("species")
    .arrange("body_mass_g desc")
    .head(1)
    .select("species", "island", "body_mass_g")
)


species,island,body_mass_g
str,str,f64
"""Adelie""","""Biscoe""",4775.0
"""Chinstrap""","""Dream""",4800.0
"""Gentoo""","""Biscoe""",6300.0


In [5]:
# Lightest 2 penguins per island
(
    penguins
    .by("island")
    .arrange("body_mass_g")
    .head(2)
    .select("island", "species", "body_mass_g")
)


island,species,body_mass_g
str,str,f64
"""Biscoe""","""Adelie""",2850.0
"""Biscoe""","""Adelie""",2850.0
"""Dream""","""Chinstrap""",2700.0
"""Dream""","""Adelie""",2900.0
"""Torgersen""","""Adelie""",2900.0
"""Torgersen""","""Adelie""",3050.0


## 4. Proportion Slicing with `prop=`

Select a fraction of rows (e.g. `prop=0.1` or `n="10%"`) overall or within each group.


In [6]:
# Top 10% heaviest penguins per species
(
    penguins
    .by("species")
    .arrange("body_mass_g desc")
    .head(prop=0.1)
    .select("species", "island", "body_mass_g")
)


species,island,body_mass_g
str,str,f64
"""Adelie""","""Biscoe""",4775.0
"""Adelie""","""Biscoe""",4725.0
"""Adelie""","""Torgersen""",4700.0
"""Adelie""","""Torgersen""",4675.0
"""Adelie""","""Dream""",4650.0
…,…,…
"""Gentoo""","""Biscoe""",5850.0
"""Gentoo""","""Biscoe""",5850.0
"""Gentoo""","""Biscoe""",5800.0


## 5. Boundary Ties (`with_ties=True`)

When multiple rows share the same value at the rank cutoff, pass `with_ties=True` to include all tied rows.


In [7]:
# Top 3 with boundary ties included
(
    penguins
    .arrange("body_mass_g desc")
    .head(3, with_ties=True)
    .select("species", "island", "body_mass_g")
)


species,island,body_mass_g
str,str,f64
"""Gentoo""","""Biscoe""",6300.0
"""Gentoo""","""Biscoe""",6050.0
"""Gentoo""","""Biscoe""",6000.0
"""Gentoo""","""Biscoe""",6000.0


## 6. Functional Style (`pt.head` & `pt.tail`)

`pt.head(df, ...)` and `pt.tail(df, ...)` are also available as standalone functional verbs:


In [8]:
# Functional top bill overall
pt.head(pt.arrange(tips, "total_bill desc"), 1)[["day", "total_bill", "tip"]]


day,total_bill,tip
cat,f64,f64
"""Sat""",50.81,10.0
